Install libraries

In [ ]:
pip install altimetry-downloader-aviso==0.3.2
pip install altimetry.io 

Load libraries

In [2]:
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
from scipy.signal import detrend
from scipy.ndimage import uniform_filter1d
import altimetry_downloader_aviso as dl_aviso
import logging
logging.basicConfig(level=logging.INFO)
from altimetry.io import AltimetryData, FileCollectionSource
import glob

Define your work directory

In [ ]:
output_dir= "/Users/ABC/Desktop/munich"


Define the region, the pass and the cycles 

In [ ]:
## Region
bbox = (0, 34, -38, 0)
localbox = [0, 34, -38, 0]

In [ ]:
##Pass and Cycles 
cycle_number = [474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578]
pass_number = [16, 1]

SWOT DOWNLOADING (You will have to enter your username and password)

In [ ]:
dl_aviso.get(
    'SWOT_L3_LR_SSH_Expert', 
    output_dir=output_dir, 
    cycle_number=cycle_number, 
    pass_number=pass_number,
)

Concatenation of the different files, The final file have this dimention : 69 x (lat_selection) x 100 (Calval duration )

In [ ]:
input_dir = "/Users/ABC/Desktop/munich/swot_exports" ## path where you have your data
output_dir = "/Users/ABC/Desktop/munich/swot_concatenated"## Saving path
os.makedirs(output_dir, exist_ok=True)

lat_min, lat_max = -38,0

for p in [1, 16]:
    files = sorted(glob.glob(os.path.join(input_dir, f"SWOT_L3_LR_SSH_Expert_*{p:03d}_*.nc")))
    
    data_list = []
    
    for i, f in enumerate(files):
        with xr.open_dataset(f) as ds:
            # Region selection 
            mask = (ds['latitude'] >= lat_min) & (ds['latitude'] <= lat_max)
            
            # Variables selection 
            subset = ds[['ssha_unfiltered', 'latitude', 'longitude', 'time']]
            subset_cut = subset.where(mask, drop=True)
            subset_cut = subset_cut.reset_coords() 
            subset_cut = subset_cut.expand_dims(dim={'cycle': [i]})
            data_list.append(subset_cut.load())
    
    if data_list:
        #Concatenation of time steps
        
        final_cube = xr.concat(data_list, dim='cycle', data_vars='all', coords=[])
        
        # save
        output_file = os.path.join(output_dir, f"swot_pass_{p}_ssha_cube.nc")
        final_cube.to_netcdf(output_file)
        
        print(f"pass {p}")
